# Продуктовая витрина по дням

Одна строка — один условный день. Витрина нужна для общего дашборда и динамики метрик.

## Что считаем

Аудиторию, события, прослушивания, уникальные треки, сессии, Listen+, повторы,
долю рекомендаций и реакции пользователей.

In [1]:
from pathlib import Path
import sys
import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SOURCE = PROJECT_ROOT / "data" / "yambda" / "flat" / "50m" / "multi_event.parquet"
MARTS = PROJECT_ROOT / "data" / "processed"
pd.set_option("display.max_columns", 30)

from src.product_mart import build_product_mart

STAGE_DB = PROJECT_ROOT / "data" / "interim" / "yambda_stage.duckdb"
MART = MARTS / "mart_product_day.parquet"
assert STAGE_DB.exists(), "Сначала выполните ноутбук 01_source_quality.ipynb"
con = duckdb.connect()

## Сборка витрины

Все новые столбцы этой витрины рассчитываются в `src/product_mart.py`.

In [2]:
build_product_mart(STAGE_DB, MART)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

{'rows': 301, 'events': 47790449, 'listens': 46467212}

## Итоговые показатели

In [3]:
summary = con.execute(f"""
SELECT
    count(*) AS days,
    sum(events) AS events,
    sum(listens) AS listens,
    median(active_users) AS median_dau,
    sum(listen_plus) * 100.0 / sum(listens) AS listen_plus_pct,
    sum(recommendation_listens) * 100.0 / sum(listens) AS recommendation_pct,
    sum(replays) * 100.0 / sum(listens) AS replay_pct,
    sum(likes) * 1000.0 / sum(listens) AS likes_per_1000,
    sum(dislikes) * 1000.0 / sum(listens) AS dislikes_per_1000,
    sum(sessions) AS sessions
FROM read_parquet('{MART.as_posix()}')
""").df().round(2)

summary.rename(columns={
    "days": "дни", "events": "события", "listens": "прослушивания",
    "median_dau": "медианный DAU", "listen_plus_pct": "Listen+, %",
    "recommendation_pct": "рекомендации, %", "replay_pct": "повторы, %",
    "likes_per_1000": "лайки на 1000", "dislikes_per_1000": "дизлайки на 1000",
    "sessions": "сессии",
})

,дни,события,прослушивания,медианный DAU,"Listen+, %","рекомендации, %","повторы, %",лайки на 1000,дизлайки на 1000,сессии
0,301,47790449.0,46467212.0,3476.0,63.21,48.34,0.47,18.97,2.32,2529801.0


## Первые пять дней

In [4]:
daily = con.execute(f"""
SELECT day_idx, active_users, events, listens, unique_tracks,
       round(listen_plus_rate * 100, 2) AS listen_plus_pct,
       round(recommendation_share * 100, 2) AS recommendation_pct,
       sessions
FROM read_parquet('{MART.as_posix()}')
ORDER BY day_idx
LIMIT 5
""").df()

daily.rename(columns={
    "day_idx": "день", "active_users": "активные пользователи",
    "events": "события", "listens": "прослушивания",
    "unique_tracks": "уникальные треки", "listen_plus_pct": "Listen+, %",
    "recommendation_pct": "рекомендации, %", "sessions": "сессии",
})

,день,активные пользователи,события,прослушивания,уникальные треки,"Listen+, %","рекомендации, %",сессии
0,0,2653,117648,114247,45449,66.23,51.49,6088
1,1,2421,109102,105991,41637,65.31,46.48,5428
2,2,2320,97376,94411,39286,62.92,47.07,4830
3,3,2591,110239,106734,42815,66.24,52.26,5900
4,4,2646,111368,107749,43184,66.49,53.18,5928


## Проверка

In [5]:
check = con.execute(f"""
SELECT count(*) = count(DISTINCT day_idx) AS unique_key,
       sum(events) = 47790449 AS events_match,
       sum(listens) = 46467212 AS listens_match,
       min(listen_plus_rate) >= 0 AND max(listen_plus_rate) <= 1 AS rates_valid
FROM read_parquet('{MART.as_posix()}')
""").df()
assert check.all(axis=None), "Проверка витрины не пройдена"
print("Проверка пройдена: ключ уникален, суммы совпадают, доли корректны.")

Проверка пройдена: ключ уникален, суммы совпадают, доли корректны.


## Вывод

Витрина готова для графиков DAU, активности, Listen+, рекомендаций, повторов и реакций.
Крайние дни перед сравнением периодов нужно проверять на полноту.